# Space Tech & AI: Manganese Prospectivity Mapping
## MOIL Limited | SIH 2026 (KHANIJ-DRISHTI)

This notebook demonstrates the *real* Space Technology and Machine Learning pipeline used to identify high-probability Manganese reserves. 

**Methodology:**
1. **Data Ingestion (Space Tech):** Connect to **Google Earth Engine (GEE)** to stream live satellite data.
2. **Feature Extraction:** Fetch Sentinel-2 (Multispectral), SRTM (Elevation/Slope), and ASTER (Geology) layers.
3. **Ground Truth Labels:** Use known MOIL active mine coordinates (Balaghat, Dongri Buzurg, etc.) as positive labels (Manganese=1) and random regional points as negative labels (Manganese=0).
4. **ML Training:** Train a `RandomForestClassifier` on the satellite spectral signatures of the host rock (Gondite).
5. **Spatial Prediction:** Generate a mineral prospectivity heatmap for the entire Nagpur-Balaghat belt and export as GeoJSON/GeoTIFF for the frontend Dashboard.

In [ ]:
!pip install earthengine-api geemap scikit-learn pandas numpy matplotlib

### 1. Authenticate and Initialize Google Earth Engine

In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Authenticate with your Google Account
# ee.Authenticate()
ee.Initialize(project="moil-predictive-intelligence")

### 2. Define the Region of Interest (ROI) & Satellite Data Layers
We are focusing on the **Sausar Group** (Central Indian Tectonic Zone) encompassing Nagpur, Bhandara, and Balaghat.

In [ ]:
# Bounding box for Balaghat/Bhandara region
roi = ee.Geometry.Rectangle([79.0, 21.3, 80.5, 22.0])

# 1. Sentinel-2 Surface Reflectance (Vegetation, Iron Oxides, Lithology)
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate("2023-01-01", "2023-12-31") \
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)) \
    .median() \
    .select(["B2", "B3", "B4", "B8", "B11", "B12"])

# 2. SRTM Digital Elevation Model (Topography, Slope)
dem = ee.Image("USGS/SRTMGL1_003").clip(roi)
slope = ee.Terrain.slope(dem)

# Combine features into a single multi-band image
satellite_features = s2.addBands(dem).addBands(slope)

### 3. Prepare Training Data (MOIL Ground Truth)
We use the exact coordinates of MOIL's operating mines. The ML model will learn the satellite "signature" of these exact locations.

In [ ]:
# Real Coordinates of MOIL Mines (Positive Samples)
moil_mines = [
    [80.1832, 21.8124], # Balaghat
    [79.7121, 21.5638], # Dongri Buzurg
    [79.2847, 21.3982], # Mansar
    [79.7523, 21.5542], # Chikla
    [79.2715, 21.4231], # Kandri
    [78.9842, 21.3812], # Gumgaon
    [79.7214, 21.6842], # Tirodi
    [80.4721, 21.9612]  # Ukwa
]

positives = ee.FeatureCollection([ee.Feature(ee.Geometry.Point(coords), {"manganese": 1}) for coords in moil_mines])

# Generate Random Negative Samples in the ROI (Areas without known mines)
negatives = ee.FeatureCollection.randomPoints(region=roi, points=100, seed=42) \
    .map(lambda f: f.set("manganese", 0))

# Combine training data
training_points = positives.merge(negatives)

### 4. Extract Satellite Signatures & Train Random Forest
We extract the actual pixel values (B2, B3, B4, Elevation, Slope) at our training points.

In [ ]:
# Sample the satellite image at the training points
training_data = satellite_features.sampleRegions(
    collection=training_points,
    properties=["manganese"],
    scale=30
)

# Convert GEE FeatureCollection to Pandas DataFrame
data_list = training_data.reduceColumns(ee.Reducer.toList(7), ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope", "manganese"]).get("list").getInfo()
df = pd.DataFrame(data_list, columns=["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope", "manganese"])

X = df.drop("manganese", axis=1)
y = df["manganese"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the ML Model
rf_model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf_model.fit(X_train, y_train)

print("Model Accuracy:", accuracy_score(y_test, rf_model.predict(X_test)))
print(classification_report(y_test, rf_model.predict(X_test)))

### 5. Generate Prospectivity Heatmap & Export
We apply the trained model back to the satellite imagery to predict manganese probability for EVERY pixel in the region.

In [ ]:
# In a real deployment, we use GEE's native SmileRandomForest for cloud inference.
gee_classifier = ee.Classifier.smileRandomForest(100).train(
    features=training_data,
    classProperty="manganese",
    inputProperties=["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]
)

# Classify the entire region
prospectivity_map = satellite_features.classify(gee_classifier)

# Visualization
Map = geemap.Map(center=[21.68, 79.9], zoom=9)
Map.addLayer(prospectivity_map, {"min": 0, "max": 1, "palette": ["blue", "cyan", "yellow", "red"]}, "Mn Prospectivity Heatmap")
Map.addLayer(positives, {"color": "purple"}, "Known MOIL Mines")
Map

# (To serve this to the Next.js frontend, we export it as a GeoJSON or GeoTIFF to our backend database)